# REE Separation for NdFeB Magnet Production

This notebook demonstrates the **difflow_ree** plugin for designing and optimizing
rare earth separation processes, specifically for producing high-purity Nd/Pr
for NdFeB permanent magnets.

## Background

NdFeB magnets are critical for:
- Electric vehicle motors
- Wind turbine generators
- Computer hard drives
- Consumer electronics

The magnetic alloy requires:
- **Nd**: Primary magnetic element (>99% purity)
- **Pr**: Often blended with Nd (didymium)
- **Dy**: Added for high-temperature performance

## Objectives

1. Model a REE separation circuit using PC88A extractant
2. Optimize for Nd/Pr recovery and purity
3. Analyze sensitivity to operating conditions
4. Estimate economics

In [1]:
import jax
import jax.numpy as jnp
from jax import grad, jacfwd

jax.config.update("jax_enable_x64", True)

# Import difflow core
from difflow.streams import make_stream, get_flows

# Import REE plugin
from difflow_ree import (
    # Database
    get_element, list_ree_elements, get_extractant,
    # Equilibrium
    REEDistribution, get_distribution_coefficient, get_separation_factor,
    # Units
    REEExtractor, REEExtractorParams,
    # Flowsheets
    ExtractScrubStripCircuit, ExtractScrubStripParams,
    # Economics
    REEPricing, estimate_capex, estimate_opex, calculate_profit,
)

print("Available REE elements:", list_ree_elements())

Available REE elements: ['La', 'Ce', 'Pr', 'Nd', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Y']


## 1. REE Feed Characterization

Typical bastnasite concentrate composition after Ce removal:

In [2]:
# Feed composition (after Ce removal)
# Flows in mol/s for a 1000 t/year plant
feed_composition = {
    "La": 0.015,   # 15%
    "Pr": 0.005,   # 5%
    "Nd": 0.020,   # 20% - main target
    "Sm": 0.002,   # 2%
    "Gd": 0.001,   # 1%
    "Dy": 0.001,   # 1% - valuable for magnets
}

print("Feed Composition (mol/s):")
print("=" * 40)
total = sum(feed_composition.values())
for elem, flow in feed_composition.items():
    props = get_element(elem)
    pct = flow / total * 100
    print(f"{elem:3s}: {flow:.4f} mol/s ({pct:5.1f}%)  - {props.group} REE")

print(f"\nTotal REE: {total:.4f} mol/s")

Feed Composition (mol/s):
La : 0.0150 mol/s ( 34.1%)  - light REE
Pr : 0.0050 mol/s ( 11.4%)  - light REE
Nd : 0.0200 mol/s ( 45.5%)  - light REE
Sm : 0.0020 mol/s (  4.5%)  - middle REE
Gd : 0.0010 mol/s (  2.3%)  - middle REE
Dy : 0.0010 mol/s (  2.3%)  - heavy REE

Total REE: 0.0440 mol/s


## 2. Extractant Selection: PC88A vs D2EHPA

PC88A is preferred for Nd/Pr separation due to higher separation factor.

In [3]:
# Compare extractants at pH 3.5
pH = 3.5
elements = ("La", "Pr", "Nd", "Sm", "Gd", "Dy")

print(f"Distribution Coefficients at pH {pH}")
print("=" * 50)
print(f"{'Element':>8} {'D2EHPA':>12} {'PC88A':>12} {'Ratio':>10}")
print("-" * 50)

for elem in elements:
    D_d2ehpa = float(get_distribution_coefficient(elem, "D2EHPA", pH))
    D_pc88a = float(get_distribution_coefficient(elem, "PC88A", pH))
    ratio = D_pc88a / D_d2ehpa
    print(f"{elem:>8} {D_d2ehpa:>12.3f} {D_pc88a:>12.3f} {ratio:>10.2f}")

# Calculate Nd/Pr separation factor using REEDistribution (pH-dependent)
dist_d2ehpa = REEDistribution(extractant="D2EHPA", elements=("Nd", "Pr"))
dist_pc88a = REEDistribution(extractant="PC88A", elements=("Nd", "Pr"))

SF_d2ehpa = float(dist_d2ehpa.get_separation_factor("Nd", "Pr", pH))
SF_pc88a = float(dist_pc88a.get_separation_factor("Nd", "Pr", pH))

print(f"\nNd/Pr Separation Factor at pH {pH}:")
print(f"  D2EHPA: {SF_d2ehpa:.2f}")
print(f"  PC88A:  {SF_pc88a:.2f} (better for Nd/Pr separation)")

Distribution Coefficients at pH 3.5
 Element       D2EHPA        PC88A      Ratio
--------------------------------------------------
      La        0.470        0.280       0.60
      Pr        3.737        5.278       1.41
      Nd        9.943       21.257       2.14
      Sm       49.831      157.580       3.16
      Gd      222.587      994.260       4.47
      Dy     1388.353     8963.962       6.46

Nd/Pr Separation Factor at pH 3.5:
  D2EHPA: 2.66
  PC88A:  4.03 (better for Nd/Pr separation)


## 3. pH Optimization for Nd/Pr Separation

Find the optimal pH to maximize Nd/Pr separation.

In [4]:
dist = REEDistribution(
    extractant="PC88A",
    elements=("La", "Pr", "Nd", "Sm", "Gd", "Dy"),
)

# Find optimal pH for Nd/Pr separation
opt_pH, max_SF = dist.optimal_pH_for_separation(
    element1="Nd",
    element2="Pr",
    pH_range=(2.0, 5.0),
)

print(f"Optimal pH for Nd/Pr separation: {opt_pH:.2f}")
print(f"Maximum separation factor: {max_SF:.2f}")

# Scan pH range
print("\npH Effect on Separation Factors:")
print(f"{'pH':>5} {'SF(Nd/Pr)':>12} {'SF(Nd/La)':>12} {'SF(Dy/Nd)':>12}")
print("-" * 45)

for pH_val in [2.0, 2.5, 3.0, 3.5, 4.0, 4.5]:
    SF_NdPr = float(dist.get_separation_factor("Nd", "Pr", pH_val))
    SF_NdLa = float(dist.get_separation_factor("Nd", "La", pH_val))
    SF_DyNd = float(dist.get_separation_factor("Dy", "Nd", pH_val))
    print(f"{pH_val:>5.1f} {SF_NdPr:>12.2f} {SF_NdLa:>12.2f} {SF_DyNd:>12.2f}")

Optimal pH for Nd/Pr separation: 5.00
Maximum separation factor: 4.79

pH Effect on Separation Factors:
   pH    SF(Nd/Pr)    SF(Nd/La)    SF(Dy/Nd)
---------------------------------------------
  2.0         3.39        38.02       125.89
  2.5         3.59        47.86       188.36
  3.0         3.80        60.26       281.84
  3.5         4.03        75.86       421.70
  4.0         4.27        95.50       630.96
  4.5         4.52       120.23       944.06


## 4. Design 3-Section Circuit for Nd/Pr Product

Extract-Scrub-Strip configuration to produce Nd+Pr (didymium) concentrate.

In [5]:
# Design circuit parameters
# With realistic D values, we need higher extraction pH for good recovery
# At pH 4.0: D_Nd ~ 200, D_La ~ 7 (good extraction with La/Nd selectivity)
# Scrubbing at pH 3.0 rejects La (D_La ~ 0.3) while retaining Nd (D_Nd ~ 4)
params = ExtractScrubStripParams(
    extractant="PC88A",
    elements=("La", "Pr", "Nd", "Sm", "Gd", "Dy"),
    target_elements=("Pr", "Nd"),  # Didymium product
    n_extraction_stages=8,
    n_scrubbing_stages=6,
    n_stripping_stages=4,
    extraction_pH=4.0,     # Higher pH for better extraction
    scrubbing_pH=3.0,      # Moderate pH to reject La but retain Nd
    stripping_pH=1.0,      # Low pH for stripping
    solvent_to_feed_ratio=1.5,
    scrub_to_solvent_ratio=0.3,
    strip_to_solvent_ratio=0.4,
)

circuit = ExtractScrubStripCircuit(params)

# Create feed stream
feed_flows = {"H2O": 50.0}  # ~1 L/s aqueous
feed_flows.update(feed_composition)

feed = make_stream(flows=feed_flows, T=298.15, P=101325.0)

# Run circuit
results = circuit(feed, T=298.15)

print("3-Section Circuit Results")
print("=" * 50)
print(f"\nOperating Conditions:")
print(f"  Extraction: pH {params.extraction_pH}, {params.n_extraction_stages} stages, S/F = {params.solvent_to_feed_ratio}")
print(f"  Scrubbing:  pH {params.scrubbing_pH}, {params.n_scrubbing_stages} stages")
print(f"  Stripping:  pH {params.stripping_pH}, {params.n_stripping_stages} stages")

3-Section Circuit Results

Operating Conditions:
  Extraction: pH 4.0, 8 stages, S/F = 1.5
  Scrubbing:  pH 3.0, 6 stages
  Stripping:  pH 1.0, 4 stages


In [6]:
# Analyze results
print("\nTarget Element Recovery (Pr, Nd):")
for elem, rec in results["target_recovery"].items():
    print(f"  {elem}: {rec*100:.1f}%")

print(f"\nProduct Purity (Pr+Nd): {results['target_purity']*100:.1f}%")

print("\nProduct Composition:")
for elem, purity in results["product_purity"].items():
    if purity > 0.001:
        print(f"  {elem}: {purity*100:.2f}%")

print("\nImpurity Rejection:")
for elem, rej in results["impurity_rejection"].items():
    print(f"  {elem}: {rej*100:.1f}% removed")


Target Element Recovery (Pr, Nd):
  Pr: 30.6%
  Nd: 80.9%

Product Purity (Pr+Nd): 81.8%

Product Composition:
  Pr: 7.08%
  Nd: 74.74%
  Sm: 8.97%
  Gd: 4.60%
  Dy: 4.62%

Impurity Rejection:
  La: 100.0% removed
  Sm: 2.9% removed
  Gd: 0.5% removed
  Dy: 0.1% removed


## 5. Sensitivity Analysis with Automatic Differentiation

Use JAX gradients to analyze sensitivity to operating parameters.

In [7]:
# Simple extractor for gradient analysis
# Use conditions that give partial recovery for meaningful gradients

def nd_recovery_fn(pH, SF_ratio, n_stages=3):
    """Calculate Nd recovery as function of operating parameters.
    
    Note: n_stages is not differentiable (discrete integer parameter).
    """
    params = REEExtractorParams(
        n_stages=n_stages,  # Fixed integer, not differentiable
        extractant="PC88A",
        elements=("La", "Nd", "Dy"),
        pH=pH,
    )
    extractor = REEExtractor(params)
    
    feed = make_stream(
        flows={"H2O": 10.0, "La": 0.015, "Nd": 0.02, "Dy": 0.001},
        T=298.15, P=101325.0,
    )
    # The solvent must name the extractant and the diluent as species, and it
    # must carry a real extractant flow: the loading capacity of the organic
    # phase is F_extractant / m (#191), so the PC88A flow of 0.0 used here
    # previously meant zero capacity and hence zero extraction. A stream whose
    # carrier is neither the extractant nor the diluent now raises instead of
    # silently defaulting the organic flow to 1.0 (#192).
    #
    # 0.5 M PC88A in kerosene is roughly 10 mol% extractant (kerosene is
    # ~0.75 g/mL and ~170 g/mol, so ~4.4 mol/L of diluent against 0.5 mol/L of
    # extractant). The total organic flow is unchanged at 10.0 * SF_ratio; it
    # is just split 10% PC88A / 90% kerosene.
    solvent = make_stream(
        flows={
            "PC88A": 1.0 * SF_ratio,
            "kerosene": 9.0 * SF_ratio,
            "La": 0.0, "Nd": 0.0, "Dy": 0.0,
        },
        T=298.15, P=101325.0,
    )
    
    _, extract, _ = extractor(feed, solvent)
    ext_flows = get_flows(extract)
    
    return ext_flows["Nd"] / 0.02

# Use conditions that give partial recovery (30-70%) for meaningful gradients
# At pH 3.0 with 3 stages and S/F=0.5, D_Nd ~ 4, giving partial recovery
pH_base = 3.0
n_stages_base = 3  # Integer, not differentiable
SF_base = 0.5

# Compute gradients (only for continuous parameters: pH and S/F ratio)
d_rec_d_pH = grad(nd_recovery_fn, argnums=0)(pH_base, SF_base, n_stages=n_stages_base)
d_rec_d_SF = grad(nd_recovery_fn, argnums=1)(pH_base, SF_base, n_stages=n_stages_base)

base_recovery = float(nd_recovery_fn(pH_base, SF_base, n_stages=n_stages_base))

print("Sensitivity Analysis for Nd Recovery")
print("=" * 50)
print(f"\nBase case: pH={pH_base}, N={n_stages_base}, S/F={SF_base}")
print(f"Recovery: {base_recovery*100:.1f}%")

print(f"\n∂(recovery)/∂(pH) = {float(d_rec_d_pH):.4f}")
print(f"  → +0.5 pH unit: {float(d_rec_d_pH)*0.5*100:+.2f}% change")

print(f"\n∂(recovery)/∂(S/F) = {float(d_rec_d_SF):.4f}")
print(f"  → +20% solvent: {float(d_rec_d_SF)*0.2*100:+.2f}% change")

# Effect of n_stages (finite difference since discrete)
rec_n3 = float(nd_recovery_fn(pH_base, SF_base, n_stages=3))
rec_n5 = float(nd_recovery_fn(pH_base, SF_base, n_stages=5))
print(f"\nEffect of stages (discrete parameter):")
print(f"  3 stages: {rec_n3*100:.1f}%")
print(f"  5 stages: {rec_n5*100:.1f}%")
print(f"  Δ(recovery) / Δ(stages) ≈ {(rec_n5-rec_n3)/(5-3)*100:.2f}% per stage")

# Verify gradient by finite difference
delta = 0.01
rec_plus = float(nd_recovery_fn(pH_base + delta, SF_base, n_stages=n_stages_base))
fd_gradient = (rec_plus - base_recovery) / delta
print(f"\nGradient verification (finite difference): {fd_gradient:.4f}")
print(f"Gradient verification (autodiff):          {float(d_rec_d_pH):.4f}")

Sensitivity Analysis for Nd Recovery

Base case: pH=3.0, N=3, S/F=0.5
Recovery: 48.3%

∂(recovery)/∂(pH) = 2.3919
  → +0.5 pH unit: +119.59% change

∂(recovery)/∂(S/F) = 0.7963
  → +20% solvent: +15.93% change

Effect of stages (discrete parameter):
  3 stages: 48.3%
  5 stages: 51.2%
  Δ(recovery) / Δ(stages) ≈ 1.42% per stage

Gradient verification (finite difference): 2.4187
Gradient verification (autodiff):          2.3919


## 6. Economic Analysis

Estimate capital and operating costs, and calculate profitability.

In [8]:
# Plant parameters
annual_capacity = 500  # tonnes REE/year

# Capital cost
capex = estimate_capex(
    annual_ree_tonnes=annual_capacity,
    n_stages_extraction=12,
    n_stages_scrubbing=8,
    n_stages_stripping=5,
    include_precipitation=True,
    year=2024,
)

print("Capital Cost Estimate")
print("=" * 40)
for item, cost in capex.items():
    print(f"{item:20s}: ${cost/1e6:,.2f} M")

print(f"\nTotal CAPEX: ${capex['total']/1e6:,.2f} M")

Capital Cost Estimate
mixer_settlers      : $1.13 M
tanks_vessels       : $0.23 M
pumps_piping        : $0.28 M
instrumentation     : $0.25 M
precipitation       : $0.18 M
ce_removal          : $0.00 M
installation        : $0.83 M
engineering         : $0.31 M
contingency         : $0.41 M
total               : $3.63 M

Total CAPEX: $3.63 M


In [9]:
# Operating cost
opex = estimate_opex(
    annual_ree_tonnes=annual_capacity,
    capex=capex["total"],
    extractant="PC88A",
)

print("Annual Operating Cost")
print("=" * 40)
for item, cost in opex.items():
    print(f"{item:20s}: ${cost/1e6:,.2f} M")

print(f"\nTotal OPEX: ${opex['total']/1e6:,.2f} M/year")

Annual Operating Cost
extractant          : $1.00 M
acid                : $0.50 M
base                : $0.25 M
precipitant         : $1.50 M
labor               : $1.68 M
utilities           : $0.24 M
maintenance         : $0.11 M
total               : $5.28 M

Total OPEX: $5.28 M/year


In [10]:
# Revenue calculation
pricing = REEPricing()

# Assume 80% of feed is Nd+Pr, 90% recovery, 95% purity
nd_production_kg = annual_capacity * 1000 * 0.5 * 0.9  # 50% Nd in product
pr_production_kg = annual_capacity * 1000 * 0.15 * 0.9  # 15% Pr
dy_production_kg = annual_capacity * 1000 * 0.02 * 0.95  # 2% Dy, separate product

nd_revenue = nd_production_kg * pricing.get_price("Nd", "99.9%", "oxide")
pr_revenue = pr_production_kg * pricing.get_price("Pr", "99.9%", "oxide")
dy_revenue = dy_production_kg * pricing.get_price("Dy", "99.9%", "oxide")

total_revenue = nd_revenue + pr_revenue + dy_revenue

print("Revenue Estimate")
print("=" * 40)
print(f"Nd ({nd_production_kg/1000:.0f} t/y): ${nd_revenue/1e6:,.2f} M")
print(f"Pr ({pr_production_kg/1000:.0f} t/y): ${pr_revenue/1e6:,.2f} M")
print(f"Dy ({dy_production_kg/1000:.0f} t/y): ${dy_revenue/1e6:,.2f} M")
print(f"\nTotal Revenue: ${total_revenue/1e6:,.2f} M/year")

Revenue Estimate
Nd (225 t/y): $35.10 M
Pr (68 t/y): $7.46 M
Dy (10 t/y): $5.56 M

Total Revenue: $48.12 M/year


In [11]:
# Profitability
profit = calculate_profit(
    revenue=total_revenue,
    opex=opex["total"],
    capex=capex["total"],
)

print("Profitability Analysis")
print("=" * 40)
print(f"Revenue:      ${profit['revenue']/1e6:>8,.2f} M/year")
print(f"OPEX:         ${profit['opex']/1e6:>8,.2f} M/year")
print(f"EBITDA:       ${profit['ebitda']/1e6:>8,.2f} M/year")
print(f"Depreciation: ${profit['depreciation']/1e6:>8,.2f} M/year")
print(f"Net Income:   ${profit['net_income']/1e6:>8,.2f} M/year")
print(f"\nPayback Period: {profit['payback_years']:.1f} years")
print(f"ROI: {profit['roi']*100:.1f}%")

Profitability Analysis
Revenue:      $   48.12 M/year
OPEX:         $    5.28 M/year
EBITDA:       $   42.84 M/year
Depreciation: $    0.36 M/year
Net Income:   $   31.86 M/year

Payback Period: 0.1 years
ROI: 878.2%


## Summary

This notebook demonstrated the **difflow_ree** plugin for:

1. **Database access** - REE properties, extractant data, separation factors
2. **Equilibrium modeling** - pH-dependent distribution coefficients
3. **Process design** - 3-section extract-scrub-strip circuit
4. **Sensitivity analysis** - Automatic differentiation for gradients
5. **Economic analysis** - CAPEX, OPEX, and profitability metrics

The differentiable framework enables:
- Rapid optimization of operating conditions
- Sensitivity analysis for process design
- Integration with gradient-based optimizers

## Next Steps

- Try different extractants (D2EHPA for heavy REE)
- Optimize number of stages and flow ratios
- Model complete separation train with Ce removal
- Perform uncertainty analysis on REE prices